# Assignment: Data Wrangling

### Reading material: `tidy_data.pdf`

**Q1.** This question provides some practice cleaning variables which have common problems.
1. For `./data/airbnb_hw.csv`, clean the `Price` variable as well as you can, and explain the choices you make. How many missing values do you end up with? (Hint: What happens to the formatting when a price goes over 999 dollars, say from 675 to 1,112?)
2. For the Minnesota police use of for data, `./data/mn_police_use_of_force.csv`, clean the `subject_injury` variable, handling the NA's; this gives a value `Yes` when a person was injured by police, and `No` when no injury occurred. What proportion of the values are missing? Is this a concern? Cross-tabulate your cleaned `subject_injury` variable with the `force_type` variable. Are there any patterns regarding when the data are missing? 

**Q2.** Go to https://sharkattackfile.net/ and download their dataset on shark attacks (Hint: `GSAF5.xls`).

1. Open the shark attack file using Pandas. It is probably not a csv file, so `read_csv` won't work.
2. Drop any columns that do not contain data.
3. Clean the year variable. Describe the range of values you see. Filter the rows to focus on attacks since 1940. Are attacks increasing, decreasing, or remaining constant over time?
4. Clean the Age variable and make a histogram of the ages of the victims.
5. What proportion of victims are male?
6. Clean the `Type` variable so it only takes three values: Provoked and Unprovoked and Unknown. What proportion of attacks are unprovoked?

---

## *Phase 1: Solution without AI

Complete all questions in the following cells without generative AI. Then commit and push the
notebook before beginning Phase 2.

**Declaration:** I completed this version without generative AI.

**Student(s):** Roshan Mahesh

**Date:** 09/06/2026

In [ ]:
# Your Phase 1 solution to the problem goes here.
# Q1.1 - Airbnb Price
import pandas as pd
 
df = pd.read_csv("./data/airbnb_hw.csv", low_memory=False)
 
price = df["Price"]
print("dtype as read:", price.dtype)
print("missing as read:", price.isna().sum())
 
# Naive conversion, to show why the column needs cleaning first.
naive = pd.to_numeric(price, errors="coerce")
print("missing if converted naively:", naive.isna().sum())
 
# Clean: strip whitespace, drop thousands separators, then convert.
clean = (
    price.astype(str)
    .str.strip()
    .str.replace(",", "", regex=False)
    .str.replace("$", "", regex=False)  # harmless here, defensive
)
df["Price"] = pd.to_numeric(clean, errors="coerce")
 
print("missing after cleaning:", df["Price"].isna().sum())
print(df["Price"].describe())
print("listings above $999:", (df["Price"] > 999).sum())
 
 # Q1.2 - MN police subject_injury
import pandas as pd
 
mn = pd.read_csv("./data/mn_police_use_of_force.csv", low_memory=False)
 
print("raw value counts:")
print(mn["subject_injury"].value_counts(dropna=False))
 
# Normalize case/whitespace, then map only clear Yes/No values.
norm = mn["subject_injury"].astype(str).str.strip().str.lower()
mn["subject_injury"] = norm.map({"yes": "Yes", "no": "No"})
 
n_missing = mn["subject_injury"].isna().sum()
print("missing:", n_missing)
print("proportion missing:", round(n_missing / len(mn), 4))
 
# Cross-tabulation with force_type, keeping missing as its own category.
xtab = pd.crosstab(
    mn["force_type"],
    mn["subject_injury"].fillna("Missing"),
    margins=True,
)
print("\ncounts:\n", xtab)
 
# Row proportions make the missingness pattern easier to read.
rates = pd.crosstab(
    mn["force_type"],
    mn["subject_injury"].fillna("Missing"),
    normalize="index",
).round(3)
print("\nrow proportions:\n", rates)


###### Question 2 ######

import re
 
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
 
# 2.1
# .xls is the legacy binary Excel format, not CSV, so read_csv fails.
# pandas needs the xlrd engine for it (openpyxl only reads .xlsx).
df = pd.read_excel("./data/GSAF5.xls", engine="xlrd")
print("shape:", df.shape)
 
 
# 2.2
print("\nnon-null counts:\n", df.notna().sum())
 
empty = [c for c in df.columns if df[c].notna().sum() == 0]
junk = [c for c in df.columns if df[c].notna().sum() < 0.01 * len(df)]
print("\nfully empty:", empty)
print("effectively empty:", junk)
 
df = df.drop(columns=junk)
print("shape after drop:", df.shape)
 
 
# 2.3
year = pd.to_numeric(df["Year"], errors="coerce")
print("\nyear range:", year.min(), "to", year.max())
print("zeros (unknown date):", (year == 0).sum())
print("before 1500:", ((year > 0) & (year < 1500)).sum())
 
df["Year"] = year.replace(0, pd.NA)  # 0 is a placeholder, not a year
 
recent = df[df["Year"] >= 1940].copy()
print("attacks since 1940:", len(recent))
 
per_year = recent["Year"].value_counts().sort_index()
print("\nattacks per decade since 1940:")
print(recent.groupby((recent["Year"] // 10 * 10).astype(int)).size())
 
plt.figure(figsize=(9, 4))
per_year.plot()
plt.title("Recorded shark attacks per year, 1940 onward")
plt.xlabel("Year")
plt.ylabel("Attacks")
plt.tight_layout()
plt.savefig("attacks_by_year.png", dpi=120)
plt.close()
 
 
# 2.4
raw_age = df["Age"].astype(str).str.strip()
 
age = pd.to_numeric(raw_age, errors="coerce")
 
# Recover decade forms: "20s", "30's", "40+" -> 20, 30, 40.
decade = raw_age.str.match(r"^\d{2}(s|'s|\+)$", na=False)
age = age.where(~decade, pd.to_numeric(raw_age.str.extract(r"^(\d{2})")[0]))
 
# Anything still non-numeric is genuinely ambiguous ("Teen", "?",
# "45 and 15", "28 & 22") and stays missing.
age = age.where((age > 0) & (age < 110))
 
df["Age"] = age
print("\nage missing:", age.isna().sum(), "of", len(df))
print(age.describe())
 
plt.figure(figsize=(8, 4))
plt.hist(age.dropna(), bins=range(0, 101, 5), edgecolor="white")
plt.title("Ages of shark attack victims")
plt.xlabel("Age")
plt.ylabel("Number of victims")
plt.tight_layout()
plt.savefig("age_histogram.png", dpi=120)
plt.close()
 
 
# 2.5
sex = df["Sex"].astype(str).str.strip().str.upper()
df["Sex"] = sex.map({"M": "M", "F": "F"})  # everything else -> NaN
counts = df["Sex"].value_counts()
print("\nsex counts:\n", counts)
print("proportion male:", round(counts["M"] / counts.sum(), 4))
 
 
# 2.6
t = df["Type"].astype(str).str.strip().str.lower()
df["Type"] = t.map({"unprovoked": "Unprovoked", "provoked": "Provoked"}).fillna(
    "Unknown"
)
tc = df["Type"].value_counts()
print("\ntype counts:\n", tc)
print("proportion unprovoked:", round(tc["Unprovoked"] / len(df), 4))
 
 


dtype as read: str
missing as read: 0
missing if converted naively: 181
missing after cleaning: 0
count    30478.000000
mean       163.589737
std        197.785454
min         10.000000
25%         80.000000
50%        125.000000
75%        195.000000
max      10000.000000
Name: Price, dtype: float64
listings above $999: 181
raw value counts:
subject_injury
NaN    9848
Yes    1631
No     1446
Name: count, dtype: int64
missing: 9848
proportion missing: 0.7619

counts:
 subject_injury               Missing    No   Yes    All
force_type                                             
Baton                              2     0     2      4
Bodily Force                    7051  1093  1286   9430
Chemical Irritant               1421   131    41   1593
Firearm                            0     2     0      2
Gun Point Display                 27    33    44    104
Improvised Weapon                 74    34    40    148
Less Lethal                       87     0     0     87
Less Lethal Projectile 

Matplotlib is building the font cache; this may take a moment.


shape: (7117, 23)

non-null counts:
 Date              7117
Year              7115
Type              7099
Country           7067
State             6630
Location          6550
Activity          6534
Name              6899
Sex               6539
Age               4123
Injury            7081
Fatal Y/N         6556
Time              3590
Species           3986
Source            7097
pdf               6799
href formula      6794
href              6796
Case Number       6798
Case Number.1     6797
original order    6799
Unnamed: 21          1
Unnamed: 22          2
dtype: int64

fully empty: []
effectively empty: ['Unnamed: 21', 'Unnamed: 22']
shape after drop: (7117, 21)

year range: 0.0 to 2026.0
zeros (unknown date): 129
before 1500: 3
attacks since 1940: 5581

attacks per decade since 1940:
Year
1940     283
1950     467
1960     618
1970     339
1980     438
1990     572
2000    1022
2010    1249
2020     593
dtype: int64

age missing: 3124 of 7117
count    3993.000000
mean       28.280

In [ ]:
#Question 1
# 1. Price is a string not a number, so that is why the formatting breaks at $1000. The prices for four digit numbers consists of a comma, which is not a valid character for a number. The solution is to clean the data by removing the commas and dollar signs, and then converting the cleaned strings to numeric values.
# 1. Missing values after cleaning: 0 out of 30,478.
# 2. The cleaning decision worth explaining: blanks are mapped to NaN, not to No. 76% of subject_injury values are missing, which is serious because it ranges from 0% for Less Lethal Projectile to 40.3% for K9 bites, 75.4% for Tasers, and 100% for Maximal Restraint Technique, so the recorded injury rates are computed on a pattern.

#Question 2
# 3. 5,581 attacks, which rise from 283 in the 1940s to 1,249 in the 2010s, so recorded attacks are increasing
# 5. 5,713 victims are male and 818 female, giving a proportion male of 0.8748
# 6. 5,263 Unprovoked, 649 Provoked, and 1,205 Unknown, so the proportion unprovoked is 0.7395


## *Phase 2: AI-assisted Revision
Keep Phase 1 frozen.

- **Phase 1 commit SHA ID:** [To be provided]
- **AI tool and model used:** [To be provided]

### *Prompts used

1. [Paste prompt]
2. [Paste follow-up prompt]

### *AI-proposed changes


[Paste the AI's concise numbered list verbatim.]

1. ...
2. ...

### *Evaluation of AI suggestions


| Change | Decision | Reason and verification |
|---|---|---|
| 1 | Accept / modify / reject | ... |
| 2 | Accept / modify / reject | ... |

### *AI-assisted solution in full
After the evaluation of AI suggestions, incorporate the accepted revisions into your Phase 1 solution and provide the final Phase 2 solution in full in the following cells.

In [ ]:
# Your Phase 2 solution to the problem goes here.


### *Reflection on AI assistance
In your own words without generative AI.

[In 150–300 words, describe the main improvements, AI mistakes or limitations, how the final solution was verified, and any remaining concerns.]